In [1]:
import pandas as pd
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
export_dir = os.getcwd()
from pathlib import Path
import pickle
from collections import defaultdict
import time
import torch
import torch.nn as nn
import copy
import torch.nn.functional as F
import optuna
import logging
import matplotlib.pyplot as plt
import ipynb
import importlib
import sys
from pathlib import Path
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from recbole.utils.case_study import full_sort_topk
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.trainer import Trainer
from recbole.utils import get_model, get_trainer


2026-04-17 14:56:18.651987: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-17 14:56:18.673785: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-17 14:56:18.673810: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-17 14:56:18.674725: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-17 14:56:18.679151: I tensorflow/core/platform/cpu_feature_guar

In [3]:
# 1. Setup Configuration


parameter_dict = {
    'model': 'SASRec',
    'dataset': 'ml-1m',
    'epochs': 2,
    'stopping_step': 10,
    'train_batch_size': 256,
    'eval_batch_size': 256,
    'hidden_size': 64,        # default is 64, try smaller if needed


    'eval_args': {
        'split': {'LS': 'valid_and_test'},   # leave-one-out
        'group_by': 'user',
        'order': 'TO',
    'mode': {'valid': 'uni100', 'test': 'uni100'}  # sample 100 negatives instead of all items
    },
    'train_neg_sample_args':None,


    'load_col': {
        'inter': ['user_id', 'item_id', 'rating', 'timestamp']
    },
}

config = Config(model='SASRec', dataset='ml-1m', config_dict=parameter_dict)
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
# 4. Initialize and Train Model
model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
trainer = get_trainer(config['trainer'], config['model'])(config, model)

# Start training
trainer.fit(train_data, valid_data, saved=True, show_progress=True)

test_result = trainer.evaluate(test_data, load_best_model=False, show_progress=True)

print("Test Results:", test_result)



Train     0:   0%|                                                         | 0/3837 [00:00<?, ?it/s]/home/mvarasteh/.conda/envs/myenv/lib/python3.10/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
Evaluate   : 100%|████████████████████| 3020/3020 [00:11<00:00, 270.59it/s, GPU RAM: 0.38 G/15.72 G]


Test Results: OrderedDict([('recall@10', 0.7654), ('mrr@10', 0.482), ('ndcg@10', 0.55), ('hit@10', 0.7654), ('precision@10', 0.0765)])


In [35]:
train_data.dataset

ml-1m
The number of users: 6041
Average actions of users: 162.5975165562914
The number of items: 3707
Average actions of items: 265.3577411510403
The number of inters: 982089
The sparsity of the dataset: 95.61449687364738%
Remain Fields: ['user_id', 'item_id', 'rating', 'timestamp', 'item_id_list', 'rating_list', 'timestamp_list', 'item_length', 'label']

In [39]:
next(iter(train_data)).item_id

tensor([ 237,  415,  123, 1761,  703,  157, 1229,  669,  436,  417, 1331,  388,
         569, 1383,  714,  601,   59,  323,  185,  670,  739,  770,  940,  536,
         147,  105,   84, 1311,  212,  336, 1564, 1323,  205,  644, 1529, 2300,
         106,  161,  210,  222, 2533,  161, 1068, 1554,  752,  144, 1014,  809,
        1558,  669,  189, 1301, 3267,  611,  974,  225,  675,  872,  121,   23,
        1613, 1578,   83, 2674,  793, 1593,  364, 2407, 1669,  123, 1132,  759,
         210,  180,  205, 1219,  556, 1257,  529,  496,  206,  292, 2173,  517,
          63, 1885,  161,  187, 1099,  333,  200,  330,  475,  131, 2678,  896,
         164,  444,  670, 1076, 2575, 1607,  738,  757, 1926,   69, 1129,  196,
         901,   27,  125,  597,   69, 1085, 1111,  791,  740, 1902, 1336, 2209,
          86,  534,   27,   46, 1217,  436, 1018,  154, 1149,  293, 1048,  322,
          14, 2396,  839,  516,  896,  734,  619,  327, 1653, 2032,  218, 2479,
        1299,  427,   58, 1022,  238,  3

In [41]:
next(iter(train_data)).item_id_list

tensor([[ 370, 2094,   42,  ...,    0,    0,    0],
        [1019,  161, 1301,  ..., 1508,  398, 1359],
        [2209, 3182, 1048,  ...,  547, 1107, 1000],
        ...,
        [ 152, 3171,  434,  ..., 1174,  147,  606],
        [  24,  106, 1323,  ...,    0,    0,    0],
        [2403, 1729, 1005,  ..., 1812, 1978, 1735]])

In [187]:
# 1. Setup Configuration

parameter_dict = {
    'model': 'SASRec',
    'dataset': 'my_dataset_ML1M', # Ensure your .inter file is in ./dataset/lastfm/
    'data_path': '/home/mvarasteh/ProvidersExplantion/dataset/ml-1m/',      # Ensure this points to the parent folder
    'epochs': 40,
    'stopping_step': 10,
    'train_batch_size': 256,
    'test_bathch_size': 256,
        'train_neg_sample_args': None,

    'eval_args': {
        'split': {'RS': [0.7, 0.1, 0.2]}, # 80% Train, 10% Valid, 10% Test
        'group_by': 'user',
        'order': 'TO',

        'mode': 'full'},

    'load_col': {
#'inter': ['user_id', 'item_id', 'rating', 'timestamp']
'inter': ['user_id', 'item_id', 'playcount', 'timestamp']

   #'item': ['item_id', 'movie_title', 'release_year', 'genre'] # 'class' is the genre in ML-1M }
    
}}
config = Config(model='SASRec', dataset='my_dataset_ML1M', config_dict=parameter_dict)

# 2. Load Dataset
dataset = create_dataset(config)
print(f"Dataset loaded: {dataset}")
train_data, valid_data, test_data = data_preparation(config, dataset)


Dataset loaded: my_dataset_ML1M
The number of users: 6041
Average actions of users: 165.49850993377484
The number of items: 3417
Average actions of items: 292.6261709601874
The number of inters: 999611
The sparsity of the dataset: 95.15741545057172%
Remain Fields: ['user_id', 'item_id', 'timestamp']


In [43]:
import torch
import numpy as np

model = model.cuda()
model.eval()

n_items = dataset.item_num

# ============================================================
# STEP 1: Generate top-10 recommendations for all test users
# ============================================================
user_topk = {}

with torch.no_grad():
    for batched_data in test_data:
        #print(len(positive_u))
        interaction, history_index, positive_u, positive_i = batched_data
        interaction = interaction.to('cuda')

        scores = model.full_sort_predict(interaction)
        scores = scores.view(-1, n_items)
        scores[:, 0] = -float('inf')  # mask padding item

        _, topk_ids = torch.topk(scores, 10, dim=1)

        uids = interaction[model.USER_ID].cpu().numpy()
        topk_ids = topk_ids.cpu().numpy()

        for uid, items in zip(uids, topk_ids):
            user_topk[uid] = items

sorted_uids = sorted(user_topk.keys())
topk_arr = np.array([user_topk[u] for u in sorted_uids])

print(f"Users: {len(sorted_uids)}, Top-k shape: {topk_arr.shape}")

# ============================================================
# STEP 2: Compute item coverage
# ============================================================
flat_items = topk_arr.flatten()
item_counts = np.bincount(flat_items, minlength=n_items)
item_counts[0] = 0  # exclude padding

# Coverage: % of items recommended at least 5 times
covered = (item_counts >= 5).sum()
coverage = covered / (n_items - 1)  # -1 to exclude padding item 0

print(f"\n--- Results ---")
print(f"Total items in dataset: {n_items - 1}")
print(f"Items recommended >= 5 times: {covered}")
print(f"Item Coverage: {coverage:.4f} ({coverage*100:.2f}%)")

# Bonus: also show coverage with min_count=1 (any appearance)
covered_any = (item_counts >= 1).sum()
coverage_any = covered_any / (n_items - 1)
print(f"Items recommended >= 1 time:  {covered_any}")
print(f"Item Coverage (>=1): {coverage_any:.4f} ({coverage_any*100:.2f}%)")

Users: 6040, Top-k shape: (6040, 10)

--- Results ---
Total items in dataset: 3706
Items recommended >= 5 times: 1471
Item Coverage: 0.3969 (39.69%)
Items recommended >= 1 time:  1971
Item Coverage (>=1): 0.5318 (53.18%)


In [190]:
from recbole.utils import get_model
import torch

# Load the checkpoint
checkpoint = torch.load('/home/mvarasteh/ProvidersExplantion/saved/sasrec_ml-1m.pth', map_location='cuda',    weights_only=False
)
# Rebuild the model from the saved config
config = checkpoint['config']
model = get_model(config['model'])(config, dataset)

# Load the trained weights
model.load_state_dict(checkpoint['state_dict'])
model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [44]:
uid_list = torch.arange(1, 10
                        ).to('cpu')

# 2. Get the Top 10 item IDs for these users
# This returns (scores, item_ids)
_, topk_iid_list = full_sort_topk(uid_list, model, test_data, k=10, device=config['device'])

/home/mvarasteh/.conda/envs/myenv/lib/python3.10/site-packages/recbole/utils/case_study.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  uid_series = torch.tensor(uid_series)


In [51]:
item_freq=dataset.inter_matrix(form='csr').sum(axis=0).A1

In [52]:

freq_norm=item_freq/item_freq.max()
ids=np.arange(len(freq_norm))

In [53]:
pop_dict=dict(zip(ids,freq_norm ))

In [54]:
sorted_pop_dict = dict(sorted(pop_dict.items(), key=lambda item: item[1], reverse=True))

In [55]:
threshold_pop=int(len(sorted_pop_dict)*0.1)

In [56]:
pop_items=list(sorted_pop_dict.keys())[0:threshold_pop]
nonpop_items=list(sorted_pop_dict.keys())[-threshold_pop:]

In [57]:
import numpy as np

rng = np.random.default_rng(42)

n_prime = 1000   # number of synthetic users (n')
M = 50           # profile length per user

# Build R_pop: each synthetic user samples M items from I_pop
R_pop = np.zeros((n_prime, dataset.item_num), dtype=np.float32)
for u in range(n_prime):
    sampled = rng.choice(pop_items, size=M, replace=True)
    R_pop[u, sampled] = 1.0

# Build R_unpop: each synthetic user samples M items from I_unpop
R_unpop = np.zeros((n_prime, dataset.item_num), dtype=np.float32)
for u in range(n_prime):
    sampled = rng.choice(nonpop_items, size=M, replace=True)
    R_unpop[u, sampled] = 1.0

print(f"R_pop shape: {R_pop.shape}, avg unique items/user: {R_pop.sum(axis=1).mean():.1f}")
print(f"R_unpop shape: {R_unpop.shape}, avg unique items/user: {R_unpop.sum(axis=1).mean():.1f}")

R_pop shape: (1000, 3707), avg unique items/user: 46.9
R_unpop shape: (1000, 3707), avg unique items/user: 46.8


In [ ]:
for batched_data in test_data:
        interaction, history_index, positive_u, positive_i = batched_data

In [201]:
print(config['eval_args'])

{'split': {'LS': 'valid_and_test'}, 'order': 'TO', 'group_by': 'user', 'mode': {'valid': 'full', 'test': 'full'}}


In [200]:
total_rows = 0
num_batches = 0

for batched_data in test_data:
    interaction, history_index, positive_u, positive_i = batched_data
    total_rows += len(interaction['user_id'])
    num_batches += 1

print(f"Total batches: {num_batches}")
print(f"Total rows: {total_rows}")
print(f"Total users in dataset: {dataset.user_num}")
print(f"Total items in dataset: {dataset.item_num}")
print(f"Rows per user: {total_rows / (dataset.user_num - 1):.1f}")

Total batches: 48
Total rows: 196296
Total users in dataset: 6041
Total items in dataset: 3417
Rows per user: 32.5


In [204]:
# Count UNIQUE users total
all_unique_users = set()
total_rows = 0

for batched_data in test_data:
    interaction, _, _, _ = batched_data
    uids = interaction['user_id'].cpu().numpy()
    all_unique_users.update(uids)
    total_rows += len(uids)

print(f"Total rows: {total_rows}")
print(f"Unique users: {len(all_unique_users)}")
print(f"Rows per unique user: {total_rows / len(all_unique_users):.1f}")

Total rows: 196296
Unique users: 6040
Rows per unique user: 32.5


In [127]:
for batched_data in test_data:
    all_batches.append(batched_data)
    interaction, history_index, positive_u, positive_i = batched_data
    user_ids = interaction[model.USER_ID].numpy()
    
    # get the unique users in order of first appearance
    _, first_occurrence = np.unique(user_ids, return_index=True)
    unique_users = user_ids[np.sort(first_occurrence)]  # [user1_id, user2_id]
    
    for pos, item in zip(positive_u.numpy(), positive_i.numpy()):
        real_uid = int(unique_users[pos])  # pos=0 → first user, pos=1 → second user
        ground_truth.setdefault(real_uid, set()).add(int(item))

print(f"Ground truth users: {len(ground_truth)}")  # should now be 6040

Ground truth users: 6040


In [130]:
ground_truth

{1: {36},
 2: {134},
 3: {208},
 4: {218},
 5: {357},
 6: {397},
 7: {151},
 8: {92},
 9: {31},
 10: {660},
 11: {821},
 12: {830},
 13: {194},
 14: {106},
 15: {520},
 16: {881},
 17: {1010},
 18: {466},
 19: {771},
 20: {642},
 21: {1168},
 22: {1196},
 23: {1297},
 24: {426},
 25: {854},
 26: {353},
 27: {719},
 28: {1300},
 29: {1459},
 30: {856},
 31: {275},
 32: {1522},
 33: {636},
 34: {92},
 35: {1648},
 36: {1072},
 37: {420},
 38: {245},
 39: {1098},
 40: {1548},
 41: {304},
 42: {778},
 43: {105},
 44: {473},
 45: {1773},
 46: {1822},
 47: {382},
 48: {361},
 49: {969},
 50: {87},
 51: {1596},
 52: {789},
 53: {246},
 54: {1237},
 55: {152},
 56: {1865},
 57: {1110},
 58: {1720},
 59: {171},
 60: {2056},
 61: {2064},
 62: {2120},
 63: {744},
 64: {877},
 65: {694},
 66: {42},
 67: {43},
 68: {20},
 69: {486},
 70: {51},
 71: {318},
 72: {157},
 73: {1149},
 74: {2056},
 75: {940},
 76: {268},
 77: {39},
 78: {829},
 79: {74},
 80: {124},
 81: {45},
 82: {434},
 83: {1905},
 

In [131]:
import torch
import numpy as np

# STEP 0
model = model.cuda()
model.eval()
for module in model.modules():
    module._forward_hooks.clear()

# STEP 1
def matrix_to_sequences(R_matrix, max_len=50):
    seqs = []
    for row in R_matrix:
        items = np.nonzero(row)[0]
        if len(items) == 0:
            items = np.array([1])
        if len(items) >= max_len:
            items = items[-max_len:]
        else:
            items = np.pad(items, (max_len - len(items), 0), constant_values=0)
        seqs.append(items)
    return np.array(seqs)

# STEP 2
def extract_activations(model, R_matrix, layer_idx=1, batch_size=32, device='cuda'):
    for module in model.modules():
        module._forward_hooks.clear()
    captured = {}
    def hook_fn(module, input, output):
        captured['act'] = output
    hook = model.trm_encoder.layer[layer_idx].register_forward_hook(hook_fn)
    sequences = matrix_to_sequences(R_matrix, max_len=50)
    all_activations = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(sequences), batch_size):
            batch_seqs = sequences[start:start + batch_size]
            seq_tensor = torch.LongTensor(batch_seqs).to(device)
            item_seq_len = (seq_tensor != 0).sum(dim=1)
            model.forward(seq_tensor, item_seq_len)
            output = captured['act']
            batch_idx = torch.arange(output.size(0), device=device)
            last_pos = (item_seq_len - 1).clamp(min=0)
            acts = output[batch_idx, last_pos, :]
            all_activations.append(acts.cpu())
    hook.remove()
    return torch.cat(all_activations, dim=0)

# STEP 3
pop_acts = extract_activations(model, R_pop, layer_idx=1)
unpop_acts = extract_activations(model, R_unpop, layer_idx=1)
steering_vector = pop_acts.mean(dim=0) - unpop_acts.mean(dim=0)
print(f"Steering vector norm: {steering_vector.norm().item():.4f}")

# STEP 4 — build ground truth FIRST, save all batches
ground_truth = {}
all_batches = []


for batched_data in test_data:
    all_batches.append(batched_data)
    interaction, history_index, positive_u, positive_i = batched_data
    user_ids = interaction[model.USER_ID].numpy()
    
    # get the unique users in order of first appearance
    _, first_occurrence = np.unique(user_ids, return_index=True)
    unique_users = user_ids[np.sort(first_occurrence)]  # [user1_id, user2_id]
    
    for pos, item in zip(positive_u.numpy(), positive_i.numpy()):
        real_uid = int(unique_users[pos])  # pos=0 → first user, pos=1 → second user
        ground_truth.setdefault(real_uid, set()).add(int(item))

print(f"Ground truth users: {len(ground_truth)}")  # should now be 6040

# STEP 5
def steered_topk(model, all_batches, dataset, steering_vector, alpha, k=10, device='cuda'):
    for module in model.modules():
        module._forward_hooks.clear()
    sv = steering_vector.to(device)
    def steering_hook(module, input, output):
        item_seq = interaction['item_id_list'].to(device)
        item_seq_len = (item_seq != 0).sum(dim=1)
        last_pos = (item_seq_len - 1).clamp(min=0)
        steered = output.clone()
        batch_idx = torch.arange(output.size(0), device=device)
        steered[batch_idx, last_pos] += alpha * sv
        return steered
    hook = model.trm_encoder.layer[1].register_forward_hook(steering_hook)
    model.eval()
    user_topk = {}
    with torch.no_grad():
        for batched_data in all_batches:
            interaction, history_index, positive_u, positive_i = batched_data
            interaction = interaction.to(device)
            scores = model.full_sort_predict(interaction)
            print(scores.shape)
            scores = scores.view(-1, dataset.item_num)
            scores[:, 0] = -float('inf')
            _, topk_ids = torch.topk(scores, k, dim=1)
            user_ids = interaction[model.USER_ID].cpu().numpy()
            topk_ids = topk_ids.cpu().numpy()
            seen = set()

            for uid, items in zip(user_ids, topk_ids):
                if uid not in seen:
                    user_topk[int(uid)] = items
                    seen.add(uid)

    hook.remove()
    sorted_uids = sorted(user_topk.keys())
    return np.array(sorted_uids), np.array([user_topk[u] for u in sorted_uids])

# STEP 6
def compute_ndcg(topk_arr, uid_arr, ground_truth, k=10):
    ndcgs = []
    for i, uid in enumerate(uid_arr):
        if uid not in ground_truth:
            continue
        rel = ground_truth[uid]
        dcg = sum(1.0 / np.log2(r + 2) for r, item in enumerate(topk_arr[i][:k]) if item in rel)
        ideal_hits = min(len(rel), k)
        idcg = sum(1.0 / np.log2(r + 2) for r in range(ideal_hits))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(ndcgs) if ndcgs else 0.0

def avg_popularity(topk_arr, item_freq):
    return item_freq[topk_arr].mean()

def item_coverage(topk_arr, n_items, min_count=5):
    flat = topk_arr.flatten()
    counts = np.bincount(flat, minlength=n_items)
    counts[0] = 0
    return (counts >= min_count).sum() / (n_items - 1)

def gini_index(topk_arr, n_items):
    flat = topk_arr.flatten()
    counts = np.bincount(flat, minlength=n_items).astype(float)
    counts[0] = 0
    counts = np.sort(counts[1:])
    n = len(counts)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * counts)) / (n * np.sum(counts)) - (n + 1) / n

# STEP 7
n_items = dataset.item_num
print(f"\n{'Alpha':>8} | {'Avg Pop@10':>12} | {'NDCG@10':>10} | {'Coverage':>10} | {'Gini':>8} | {'Users':>6}")
print("-" * 72)
for alpha in [2, 1, 0.5, 0.2, 0.1, 0, -0.1, -0.2, -0.5, -1.0, -2.0 ]:
    uids, topk = steered_topk(model, all_batches, dataset, steering_vector, alpha=alpha, k=10)
    print(f"{alpha:>+8.1f} | {avg_popularity(topk, item_freq):>12.4f} | {compute_ndcg(topk, uids, ground_truth):>10.4f} | {item_coverage(topk, n_items):>9.4f} | {gini_index(topk, n_items):>8.4f} | {len(uids):>6}")

Steering vector norm: 14.2144
Ground truth users: 6040

   Alpha |   Avg Pop@10 |    NDCG@10 |   Coverage |     Gini |  Users
------------------------------------------------------------------------
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Size([202, 3707])
torch.Siz

KeyboardInterrupt: 

In [92]:
all_batches[3]

(The batch_size of interaction: 202
     user_id, torch.Size([202]), cpu, torch.int64
     item_id, torch.Size([202]), cpu, torch.int64
     item_id_list, torch.Size([202, 50]), cpu, torch.int64
     rating, torch.Size([202]), cpu, torch.float32
     rating_list, torch.Size([202, 50]), cpu, torch.float32
     timestamp, torch.Size([202]), cpu, torch.float32
     item_length, torch.Size([202]), cpu, torch.int64
     timestamp_list, torch.Size([202, 50]), cpu, torch.float32
     label, torch.Size([202]), cpu, torch.float32
 ,
 tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,